In [4]:
# =======================
# IEOR4004 Project – Q1 
# =======================

import pandas as pd
import numpy as np
from gurobipy import Model, GRB, quicksum

# ---------- 1. Load datasets ----------
base = "/Users/cuilinnan/Desktop/ChildCareDeserts_Data/"
df_fac = pd.read_csv(base + "child_care_regulated.csv")
df_inc = pd.read_csv(base + "avg_individual_income.csv")
df_pop = pd.read_csv(base + "population.csv")
df_emp = pd.read_csv(base + "employment_rate.csv")
df_loc = pd.read_csv(base + "potential_locations.csv")

# ---------- 2. Standardize ZIP codes ----------
df_fac.loc[df_fac["zip_code"] >= 100000, "zip_code"] = df_fac["zip_code"] // 10000
df_inc.loc[df_inc["ZIP code"] >= 100000, "ZIP code"] = df_inc["ZIP code"] // 10000
df_pop.loc[df_pop["zipcode"] >= 100000, "zipcode"] = df_pop["zipcode"] // 10000
df_emp.loc[df_emp["zipcode"] >= 100000, "zipcode"] = df_emp["zipcode"] // 10000
df_loc.loc[df_loc["zipcode"] >= 100000, "zipcode"] = df_loc["zipcode"] // 10000

# ---------- 3. Compute existing capacities ----------
df_fac["existing_capacity_0_12"] = (
    df_fac[["infant_capacity","toddler_capacity","preschool_capacity"]].sum(axis=1)
    + (5/12) * df_fac["children_capacity"]
)
df_fac["existing_capacity_0_5"] = df_fac[["infant_capacity","toddler_capacity"]].sum(axis=1)

cap_by_zip = (
    df_fac.groupby("zip_code")[["existing_capacity_0_12","existing_capacity_0_5"]]
    .sum().reset_index()
)

# ---------- 4. Population adjustment ----------
df_pop["pop_0_5"] = df_pop["-5"]
df_pop["10-12"]   = df_pop["10-14"] * 3/5
df_pop["pop_0_12"] = df_pop[["-5","5-9","10-12"]].sum(axis=1)
pop_by_zip = df_pop[["zipcode","pop_0_5","pop_0_12"]].rename(columns={"zipcode":"zip_code"})

# ---------- 5. Income & employment ----------
inc_by_zip = df_inc.rename(columns={"ZIP code":"zip_code"})
emp_by_zip = df_emp.rename(columns={"zipcode":"zip_code"})

# ---------- 6. Merge ----------
df_zip = (
    cap_by_zip
    .merge(pop_by_zip, on="zip_code", how="outer")
    .merge(inc_by_zip, on="zip_code", how="outer")
    .merge(emp_by_zip, on="zip_code", how="outer")
)

# ---------- 7. Cleaning ----------
nonzip_cols = [c for c in df_zip.columns if c != "zip_code"]
df_zip = df_zip.dropna(how="all", subset=nonzip_cols)
need_cols = ["existing_capacity_0_12","existing_capacity_0_5","pop_0_12","pop_0_5"]
df_zip = df_zip.dropna(subset=need_cols)

# ---------- 8. High-demand classification ----------
df_zip["high_demand"] = (df_zip["average income"] <= 60000) | (df_zip["employment rate"] >= 0.6)

# ---------- 9. Minimum required slots ----------
df_zip["min_required_0_12"] = np.where(
    df_zip["high_demand"],
    0.5 * df_zip["pop_0_12"],
    (1/3) * df_zip["pop_0_12"]
)
df_zip["min_required_0_5"] = (2/3) * df_zip["pop_0_5"]

# ---------- 10. Identify deserts ----------
df_zip["is_desert_0_12"] = df_zip["existing_capacity_0_12"] <= df_zip["min_required_0_12"]
df_zip["is_desert_0_5"]  = df_zip["existing_capacity_0_5"]  <= df_zip["min_required_0_5"]
df_zip["is_desert_any"]  = df_zip["is_desert_0_12"] | df_zip["is_desert_0_5"]
df_zip_opt = df_zip[df_zip["is_desert_any"]].copy()
print(f"Desert ZIPs: {df_zip_opt.shape[0]} / {df_zip.shape[0]}")

# ---------- 11. Parameters ----------
ZIPS = list(df_zip_opt["zip_code"].astype(int))
req_012 = df_zip_opt.set_index("zip_code")["min_required_0_12"].to_dict()
req_05  = df_zip_opt.set_index("zip_code")["min_required_0_5"].to_dict()

types       = ["small","medium","large"]
capacity    = {"small":100, "medium":200, "large":400}
capacity_05 = {"small":50, "medium":100, "large":200}
cost_build  = {"small":65000, "medium":95000, "large":115000}

# === Facility-level sets & dicts ===
F_tab = df_fac[df_fac["zip_code"].isin(ZIPS)][
    ["facility_id","zip_code","existing_capacity_0_12","existing_capacity_0_5"]
].copy()
F = list(F_tab["facility_id"])
zip_of_f = dict(zip(F_tab["facility_id"], F_tab["zip_code"]))
n12_f = dict(zip(F_tab["facility_id"], F_tab["existing_capacity_0_12"]))
n05_f = dict(zip(F_tab["facility_id"], F_tab["existing_capacity_0_5"]))

exist_zip_012 = F_tab.groupby("zip_code")["existing_capacity_0_12"].sum().to_dict()
exist_zip_005 = F_tab.groupby("zip_code")["existing_capacity_0_5"].sum().to_dict()
for z in ZIPS:
    exist_zip_012.setdefault(z, 0.0)
    exist_zip_005.setdefault(z, 0.0)

# ---------- 12. Build model ----------
m = Model("Q1_facility_level_v3")

# --- decision variables ---
# New builds (by ZIP)
y_build   = m.addVars(ZIPS, types, vtype=GRB.CONTINUOUS, lb=0.0, name="build")
z05_build = m.addVars(ZIPS, types, vtype=GRB.CONTINUOUS, lb=0.0, name="slots05_build")

# Facility-level expansion (piecewise as "added" capacity)
x1 = m.addVars(F, vtype=GRB.CONTINUOUS, lb=0.0, name="expand_add_0_100")   # up to +100% (added ≤ n_f)
x2 = m.addVars(F, vtype=GRB.CONTINUOUS, lb=0.0, name="expand_add_100_120") # extra +0–20% (added ≤ 0.2*n_f)
b  = m.addVars(F, vtype=GRB.CONTINUOUS, lb=0.0, ub=1.0, name="trigger_over100")
z05_expand = m.addVars(F, vtype=GRB.CONTINUOUS, lb=0.0, name="slots05_from_expansion")

# --- capacity bounds for 0–5 allocation ---
for z in ZIPS:
    for t in types:
        m.addConstr(z05_build[z,t] <= capacity_05[t]*y_build[z,t], name=f"limit05_build_{z}_{t}")
for f in F:
    m.addConstr(z05_expand[f] <= x1[f] + x2[f], name=f"limit05_expand_{f}")

# --- piecewise expansion caps (added capacity): 0–100%, 100–120% ---
for f in F:
    nf = float(n12_f[f])     # original base capacity
    u1 = 1.0 * nf            # first segment: added ≤ n_f  (final ≤ 2.0×n_f)
    u2 = 0.2 * nf            # second segment: added ≤ 0.2*n_f (final ≤ 2.2×n_f)

    m.addConstr(x1[f] <= u1, name=f"x1cap_{f}")
    m.addConstr(x2[f] <= u2, name=f"x2cap_{f}")
    # enter second segment ⇒ trigger one-time baseline fee
    m.addConstr(x2[f] <= u2 * b[f], name=f"trigger_{f}")

# --- coverage constraints ---
for z in ZIPS:
    expand12_z = quicksum(x1[f] + x2[f] for f in F if zip_of_f[f]==z)
    new12_z    = quicksum(capacity[t]*y_build[z,t] for t in types)
    m.addConstr(exist_zip_012[z] + expand12_z + new12_z >= req_012[z], name=f"cov012_{z}")

    new05_z = quicksum(z05_build[z,t] for t in types)
    exp05_z = quicksum(z05_expand[f] for f in F if zip_of_f[f]==z)
    m.addConstr(exist_zip_005[z] + new05_z + exp05_z >= req_05[z], name=f"cov05_{z}")

# ---------- 13. Objective ----------
# New build cost + equipment ($100/slot for both new & expansion 0–5)
build_cost = quicksum(cost_build[t]*y_build[z,t] for z in ZIPS for t in types)
equip_cost = (
    quicksum(100*z05_build[z,t] for z in ZIPS for t in types) +
    quicksum(100*z05_expand[f] for f in F)
)

# Expansion variable cost + one-time baseline fee if >100%
var_exp_cost = quicksum(
    (20000 + 200*n12_f[f])*(x1[f]/max(1.0,n12_f[f])) +
    (20000 + 200*n12_f[f])*(x2[f]/max(1.0,n12_f[f]))
    for f in F
)
baseline_fee = quicksum((20000 + 200*n12_f[f])*b[f] for f in F)

m.setObjective(build_cost + equip_cost + var_exp_cost + baseline_fee, GRB.MINIMIZE)

# ---------- 14. Solve LP then MIP ----------
print(" Solving LP relaxation...")
m.optimize()
lp_obj = m.objVal
print(f"[LP] Objective = {lp_obj:,.2f}")

# Integerize (warm start from LP)
for var in y_build.values():   var.vType = GRB.INTEGER; var.Start = round(var.X)
for var in z05_build.values(): var.vType = GRB.INTEGER; var.Start = round(var.X)
for var in x1.values():        var.vType = GRB.INTEGER; var.Start = round(var.X)
for var in x2.values():        var.vType = GRB.INTEGER; var.Start = round(var.X)
for var in z05_expand.values():var.vType = GRB.INTEGER; var.Start = round(var.X)
for var in b.values():         var.vType = GRB.BINARY;  var.Start = 1 if var.X>1e-6 else 0

m.update()
m.setParam("MIPFocus",1)
m.setParam("Heuristics",0.2)
m.setParam("Threads",0)
print(" Solving MIP integer model...")
m.optimize()
mip_obj = m.objVal
print(f"[MIP] Objective = {mip_obj:,.2f}")

# ---------- 15. Export results ----------
rows=[]
for z in ZIPS:
    exp12 = sum((x1[f].X + x2[f].X) for f in F if zip_of_f[f]==z)
    new12 = sum(capacity[t]*y_build[z,t].X for t in types)
    new05 = sum(z05_build[z,t].X for t in types)
    exp05 = sum(z05_expand[f].X       for f in F if zip_of_f[f]==z)
    rows.append({
        "zip": int(z),
        "expand_0_12": exp12,
        "new_0_12":    new12,
        "new_0_5":     new05,
        "expand_0_5":  exp05,
        "cov012_LHS":  exist_zip_012[z] + exp12 + new12,
        "cov012_req":  req_012[z],
        "cov05_LHS":   exist_zip_005[z] + new05 + exp05,
        "cov05_req":   req_05[z],
    })

df_res = pd.DataFrame(rows).round(2)
out_path = "/Users/cuilinnan/Desktop/scenario1_solution_by_zip_facility_level.xlsx"
df_res.to_excel(out_path, index=False)
print(f" Results saved to {out_path}")
print(df_res.head())


Desert ZIPs: 1036 / 1066
🔹 Solving LP relaxation...
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (mac64[arm] - Darwin 24.6.0 24G90)

CPU model: Apple M4
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Optimize a model with 64180 rows, 65216 columns and 157319 nonzeros
Model fingerprint: 0x5ec14504
Coefficient statistics:
  Matrix range     [2e-01, 4e+02]
  Objective range  [1e+02, 1e+05]
  Bounds range     [1e+00, 1e+00]
  RHS range        [2e-01, 1e+04]
Presolve removed 50546 rows and 27340 columns
Presolve time: 0.05s
Presolved: 13634 rows, 37876 columns, 67058 nonzeros

Concurrent LP optimizer: primal simplex, dual simplex, and barrier
Showing barrier log only...

Ordering time: 0.00s

Barrier statistics:
 AA' NZ     : 3.344e+03
 Factor NZ  : 1.299e+04 (roughly 4 MB of memory)
 Factor Ops : 8.406e+04 (less than 1 second per iteration)
 Threads    : 1

                  Objective                Residual
Iter       Primal          Dual         Prima

Task 2 Part 1 Data Loader

In [ ]:
import pandas as pd
import numpy as np
import math
from gurobipy import Model, GRB, quicksum

class DataLoader:
    """负责加载和预处理数据的类，将结果存入CSV文件
    Class responsible for loading and preprocessing data, saving results to CSV files"""
    def __init__(self, data_dir=""):
        """
        初始化DataLoader
        Initialize DataLoader
        :param data_dir: 数据文件所在目录
                         Data directory where files are located
        """
        self.data_dir = data_dir
        self.df_zip = None
        self.df_potential = None
        self.processed_data = None
        self.facility_data = None  # 新增：存储每个设施的详细信息
        self.location_distances = None
        self.facility_locations = None  # 存储现有设施位置
                                      # Store existing facility locations
        self.too_close_positions = None  # 存储与现有设施太近的位置
                                       # Store positions too close to existing facilities
        self.facility_zip_map = None   # 新增：设施与zip code的映射

    def load_and_process(self, output_file="processed_data.csv", 
                         distance_file="location_distances.csv",
                         facility_locations_file="existing_facilities_locations.csv",
                         too_close_file="too_close_positions.csv",
                         facility_data_file="facility_data.csv"):
        """
        加载并处理所有数据，保存到CSV文件
        Load and process all data, save to CSV files
        :param output_file: 处理后的数据输出文件名
                            Output file name for processed data
        :param distance_file: 位置距离数据输出文件名
                              Output file name for location distance data
        :param facility_locations_file: 现有设施位置输出文件名
                                        Output file name for existing facility locations
        :param too_close_file: 太近位置输出文件名
                               Output file name for too close positions
        :param facility_data_file: 设施详细数据输出文件名
                                   Output file name for facility detailed data
        :return: 处理后的数据DataFrame
                 Processed data DataFrame
        """
        # 加载原始数据
        # Load raw data
        self._load_data()
        # 预处理数据
        # Preprocess data
        self._preprocess_data()
        # 保存处理后的数据
        # Save processed data
        self._save_processed_data(output_file, distance_file, facility_locations_file, 
                                 too_close_file, facility_data_file)
        return self.processed_data

    def _load_data(self):
        """加载所有原始数据文件
        Load all raw data files"""
        # 加载托儿机构数据
        # Load childcare facility data
        df_fac = pd.read_csv(f"{self.data_dir}child_care_regulated.csv")
        # 为每个facility添加唯一ID
        df_fac["facility_id"] = range(len(df_fac))
        
        # 加载收入数据
        # Load income data
        df_inc = pd.read_csv(f"{self.data_dir}avg_individual_income.csv")
        # 加载人口数据
        # Load population data
        df_pop = pd.read_csv(f"{self.data_dir}population.csv")
        # 加载就业率数据
        # Load employment rate data
        df_emp = pd.read_csv(f"{self.data_dir}employment_rate.csv")
        # 加载潜在位置数据
        # Load potential location data
        self.df_potential = pd.read_csv(f"{self.data_dir}potential_locations.csv")
        # 处理ZIP码格式
        # Process ZIP code format
        self._process_zip_codes(df_fac, df_inc, df_pop, df_emp, self.df_potential)
        # 计算容量
        # Calculate capacities
        self._calculate_capacities(df_fac)
        # 计算人口
        # Calculate population
        self._calculate_population(df_pop)
        # 合并数据
        # Merge data
        self.df_zip = self._merge_data(df_fac, df_inc, df_pop, df_emp)
        # 删除关键字段缺失的行
        # Remove rows with missing key fields
        self.df_zip = self.df_zip.dropna(subset=["existing_capacity_0_12", "pop_0_12", "pop_0_5"])
        # 保存现有设施位置
        # Save existing facility locations
        self.facility_locations = self._extract_facility_locations(df_fac)
        # 保存设施详细数据
        # Save facility detailed data
        self.facility_data = df_fac.copy()
        # 创建设施与zip code的映射
        # Create facility to zip code mapping
        self.facility_zip_map = df_fac[["facility_id", "zip_code"]].copy()

    def _extract_facility_locations(self, df_fac):
        """提取现有设施的位置信息
        Extract location information of existing facilities"""
        # 确保zip_code是整数类型
        # Ensure zip_code is integer type
        df_fac["zip_code"] = df_fac["zip_code"].astype(int)
        # 只保留有经纬度数据的行
        # Keep only rows with latitude and longitude data
        df_fac = df_fac.dropna(subset=["latitude", "longitude"])
        # 选择需要的列
        # Select required columns
        facility_locations = df_fac[["facility_id", "zip_code", "latitude", "longitude"]].copy()
        return facility_locations

    def _process_zip_codes(self, *dfs):
        """标准化ZIP码格式
        Standardize ZIP code format"""
        for df in dfs:
            if df is None:
                continue
            if "zip_code" in df.columns:
                df.loc[df["zip_code"] >= 100000, "zip_code"] = df["zip_code"] // 10000
            elif "ZIP code" in df.columns:
                df.loc[df["ZIP code"] >= 100000, "ZIP code"] = df["ZIP code"] // 10000
            elif "zipcode" in df.columns:
                df.loc[df["zipcode"] >= 100000, "zipcode"] = df["zipcode"] // 10000

    def _calculate_capacities(self, df_fac):
        """计算托儿机构容量
        Calculate childcare facility capacities"""
        df_fac["existing_capacity_0_12"] = (
            df_fac[["total_capacity"]]
        )
        df_fac["existing_capacity_0_5"] = df_fac[["infant_capacity", "toddler_capacity"]].sum(axis=1)
        # 新增：为每个设施创建初始容量字段
        # New: Create initial capacity fields for each facility
        df_fac["initial_capacity_0_12"] = df_fac["existing_capacity_0_12"]
        df_fac["initial_capacity_0_5"] = df_fac["existing_capacity_0_5"]

    def _calculate_population(self, df_pop):
        """计算人口统计数据
        Calculate population statistics"""
        df_pop["pop_0_5"] = df_pop["-5"]
        df_pop["10-12"] = df_pop["10-14"] * (3/5)
        df_pop["pop_0_12"] = df_pop[["-5", "5-9", "10-12"]].sum(axis=1)

    def _merge_data(self, df_fac, df_inc, df_pop, df_emp):
        """合并所有数据集
        Merge all datasets"""
        cap_by_zip = (
            df_fac.groupby("zip_code")[["existing_capacity_0_12", "existing_capacity_0_5"]]
            .sum()
            .reset_index()
        )
        pop_by_zip = df_pop[["zipcode", "pop_0_5", "pop_0_12"]].rename(columns={"zipcode": "zip_code"})
        inc_by_zip = df_inc.rename(columns={"ZIP code": "zip_code"})
        emp_by_zip = df_emp.rename(columns={"zipcode": "zip_code"})
        df_zip = (
            cap_by_zip
            .merge(pop_by_zip, on="zip_code", how="outer")
            .merge(inc_by_zip, on="zip_code", how="outer")
            .merge(emp_by_zip, on="zip_code", how="outer")
        )
        return df_zip

    def _preprocess_data(self):
        """预处理数据，包括计算需求和处理潜在位置
        Preprocess data, including calculating demand and processing potential locations"""
        # 计算需求
        # Calculate demand
        self._calculate_requirements()
        # 处理潜在位置数据
        # Process potential location data
        self._process_potential_locations()
        # 计算与现有设施太近的位置
        # Calculate positions too close to existing facilities
        self._find_too_close_positions()
        # 将处理后的数据整理为便于优化的形式
        # Format processed data for optimization
        self._format_processed_data()

    def _calculate_requirements(self):
        """计算最低需求
        Calculate minimum requirements"""
        self.df_zip["high_demand"] = (self.df_zip["average income"] <= 60000) | (self.df_zip["employment rate"] >= 0.6)
        self.df_zip["min_required_0_12"] = self.df_zip.apply(
            lambda r: 0.5 * r["pop_0_12"] if r["high_demand"] else (1/3) * r["pop_0_12"],
            axis=1
        )
        self.df_zip["min_required_0_5"] = (2/3) * self.df_zip["pop_0_5"]

    def _process_potential_locations(self):
        """处理潜在位置数据，计算位置之间的距离
        Process potential location data, calculate distances between locations"""
        # 确保没有重复的位置
        # Ensure no duplicate positions
        self.df_potential = self.df_potential.drop_duplicates(subset=["zipcode", "latitude", "longitude"])
        # 为每个ZIP码内的位置分配唯一ID
        # Assign unique ID to positions within each ZIP code
        self.df_potential["location_id"] = self.df_potential.groupby("zipcode").cumcount()
        # 创建位置距离DataFrame
        # Create location distance DataFrame
        location_distances = []
        # 按ZIP码分组处理
        # Process by ZIP code grouping
        zip_groups = self.df_potential.groupby("zipcode")
        for zip_code, group in zip_groups:
            if len(group) > 1:
                # 计算Haversine距离矩阵
                # Calculate Haversine distance matrix
                coords = group[["latitude", "longitude"]].values
                distances = self._haversine_distance_matrix(coords)
                # 转换为长格式
                # Convert to long format
                for i in range(len(group)):
                    for j in range(i+1, len(group)):
                        location_distances.append({
                            "zip_code": zip_code,
                            "loc1_id": group.iloc[i]["location_id"],
                            "loc2_id": group.iloc[j]["location_id"],
                            "distance": distances[i, j],
                            "too_close": distances[i, j] < 0.06
                        })
        self.location_distances = pd.DataFrame(location_distances)

    def _haversine_distance_matrix(self, coords):
        """
        计算坐标矩阵中所有点对之间的距离（英里）
        Calculate distances between all point pairs in coordinate matrix (miles)
        :param coords: 二维数组，每行是[latitude, longitude]
                       2D array, each row is [latitude, longitude]
        :return: 距离矩阵
                 Distance matrix
        """
        n = len(coords)
        dist_matrix = np.zeros((n, n))
        # 地球半径（英里）
        # Earth radius (miles)
        R = 3959
        for i in range(n):
            for j in range(i+1, n):
                lat1, lon1 = coords[i]
                lat2, lon2 = coords[j]
                # 转换为弧度
                # Convert to radians
                lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
                # Haversine公式
                # Haversine formula
                dlat = lat2 - lat1
                dlon = lon2 - lon1
                a = math.sin(dlat/2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon/2)**2
                c = 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))
                dist = R * c
                dist_matrix[i, j] = dist
                dist_matrix[j, i] = dist
        return dist_matrix

    def _find_too_close_positions(self):
        """计算哪些潜在位置与现有设施太近
        Calculate which potential positions are too close to existing facilities"""
        too_close_positions = []
        # 按ZIP码分组处理
        # Process by ZIP code grouping
        zip_groups = self.df_potential.groupby("zipcode")
        for zip_code, group in zip_groups:
            # 获取该ZIP码的现有设施位置
            # Get existing facility locations for this ZIP code
            facilities = self.facility_locations[self.facility_locations["zip_code"] == zip_code]
            if facilities.empty:
                continue
            # 获取该ZIP码的潜在位置
            # Get potential locations for this ZIP code
            coords = group[["latitude", "longitude"]].values
            facility_coords = facilities[["latitude", "longitude"]].values
            # 对每个潜在位置，检查是否与任何现有设施太近
            # For each potential location, check if it's too close to any existing facility
            for i in range(len(group)):
                for j in range(len(facilities)):
                    dist = self._haversine_distance(
                        (coords[i][0], coords[i][1]),
                        (facility_coords[j][0], facility_coords[j][1])
                    )
                    if dist < 0.06:  # 小于0.06英里
                                     # Less than 0.06 miles
                        too_close_positions.append({
                            "zip_code": zip_code,
                            "location_id": group.iloc[i]["location_id"],
                            "facility_id": facilities.iloc[j]["facility_id"],
                            "too_close_to_facility": True
                        })
        self.too_close_positions = pd.DataFrame(too_close_positions)

    def _haversine_distance(self, coord1, coord2):
        """
        计算两点之间的Haversine距离（英里）
        Calculate Haversine distance between two points (miles)
        :param coord1: (latitude, longitude)
        :param coord2: (latitude, longitude)
        :return: 距离（英里）
                 Distance (miles)
        """
        lat1, lon1 = coord1
        lat2, lon2 = coord2
        # 地球半径（英里）
        # Earth radius (miles)
        R = 3959
        # 转换为弧度
        # Convert to radians
        lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
        # Haversine公式
        # Haversine formula
        dlat = lat2 - lat1
        dlon = lon2 - lon1
        a = math.sin(dlat/2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon/2)**2
        c = 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))
        dist = R * c
        return dist

    def _format_processed_data(self):
        """将处理好的数据整理成适合优化的格式
        Format processed data into a format suitable for optimization"""
        # 创建包含所有必要信息的DataFrame
        # Create DataFrame containing all necessary information
        processed = self.df_zip.copy()
        # 添加现有设施数量
        # Add existing facility count
        fac_count = self.df_zip["zip_code"].map(
            self.df_zip.groupby("zip_code")["existing_capacity_0_12"].count()
        )
        processed["existing_facilities"] = fac_count
        # 添加潜在位置信息
        # Add potential location information
        loc_count = self.df_potential.groupby("zipcode").size().reset_index(name="potential_locations")
        processed = processed.merge(loc_count, left_on="zip_code", right_on="zipcode", how="left")
        processed["potential_locations"] = processed["potential_locations"].fillna(0).astype(int)
        # 为每个ZIP码添加扩容上限（20%）
        # Add expansion upper bound for each ZIP code (20%)
        processed["expand_upper_bound"] = processed["existing_capacity_0_12"] * 0.2
        self.processed_data = processed

    def _save_processed_data(self, output_file, distance_file, facility_locations_file, too_close_file, facility_data_file):
        """保存处理好的数据到CSV文件
        Save processed data to CSV files"""
        if self.processed_data is not None:
            self.processed_data.to_csv(output_file, index=False)
            print(f"Processed data saved to {output_file}")
        if self.location_distances is not None and not self.location_distances.empty:
            self.location_distances.to_csv(distance_file, index=False)
            print(f"Location distances saved to {distance_file}")
        if self.facility_locations is not None and not self.facility_locations.empty:
            self.facility_locations.to_csv(facility_locations_file, index=False)
            print(f"Facility locations saved to {facility_locations_file}")
        if self.too_close_positions is not None and not self.too_close_positions.empty:
            self.too_close_positions.to_csv(too_close_file, index=False)
            print(f"Too close positions saved to {too_close_file}")
        if self.facility_data is not None and not self.facility_data.empty:
            self.facility_data.to_csv(facility_data_file, index=False)
            print(f"Facility detailed data saved to {facility_data_file}")
        if self.facility_zip_map is not None and not self.facility_zip_map.empty:
            self.facility_zip_map.to_csv("facility_zip_map.csv", index=False)
            print("Facility zip code mapping saved to facility_zip_map.csv")

# 主程序
# Main program
if __name__ == "__main__":
    print("Starting data preprocessing...")
    # 开始数据预处理
    data_loader = DataLoader(data_dir="./")
    processed_data = data_loader.load_and_process()
    print("Data preprocessing completed.")
    # 数据预处理完成

Task 2 Part 2 Optimizer

In [ ]:
import pandas as pd
from gurobipy import Model, GRB, quicksum

class RealisticCapacityPlanner:
    """Optimizer for solving realistic capacity expansion and location problems (per facility expansion)"""
    def __init__(self, data_file="processed_data.csv",
                 distance_file="location_distances.csv",
                 too_close_file="too_close_positions.csv",
                 facility_data_file="facility_data.csv",
                 facility_zip_map_file="facility_zip_map.csv"):
        # Data file paths
        self.data_file = data_file
        self.distance_file = distance_file
        self.too_close_file = too_close_file
        self.facility_data_file = facility_data_file
        self.facility_zip_map_file = facility_zip_map_file
        self.model = None
        self.data = None
        self.distances = None
        self.too_close = None
        self.facility_data = None
        self.facility_zip_map = None
        self.zips = None
        self.facilities = None  # Store all facility IDs
        self.potential_locations = None
        # Facility types
        self.facility_types = ["small", "medium", "large"]
        # Segment bounds (fraction)
        self.segment_bounds = [
            (0.0, 0.10),   # 0 - 10%
            (0.10, 0.15),  # 10 - 15%
            (0.15, 0.20)   # 15 - 20%
        ]
    
    def load_data(self):
        """Load data and perform basic preprocessing"""
        self.data = pd.read_csv(self.data_file)
        self.facility_data = pd.read_csv(self.facility_data_file)
        self.facility_zip_map = pd.read_csv(self.facility_zip_map_file)
        
        # Ensure zip_code is integer
        self.data["zip_code"] = self.data["zip_code"].astype(int)
        self.facility_data["zip_code"] = self.facility_data["zip_code"].astype(int)
        self.facility_zip_map["zip_code"] = self.facility_zip_map["zip_code"].astype(int)
        
        # Get all zip codes
        self.zips = sorted(self.data["zip_code"].unique())
        
        # Get all facility IDs
        self.facilities = self.facility_data["facility_id"].tolist()
        
        # Load distance and too-close position tables
        self.distances = pd.read_csv(self.distance_file)
        self.too_close = pd.read_csv(self.too_close_file)
        
        # Ensure zip fields in these tables are integers (if exist)
        if "zip_code" in self.distances.columns:
            self.distances["zip_code"] = self.distances["zip_code"].astype(int)
        if "zip_code" in self.too_close.columns:
            self.too_close["zip_code"] = self.too_close["zip_code"].astype(int)
        
        # Build potential_locations dict
        self.potential_locations = {}
        for z in self.zips:
            # Find the number of potential locations for this zip from self.data
            row = self.data[self.data["zip_code"] == z].iloc[0]
            num_locs = int(row["potential_locations"]) if "potential_locations" in row.index else 0
            self.potential_locations[z] = list(range(num_locs))

    def build_model(self):
        # If data is not loaded, load it first
        if self.data is None:
            self.load_data()
        self.model = Model("realistic_capacity_planning_per_facility")
        self.model.setParam('OutputFlag', 0)  # Default to turn off output, user can turn it on in solve()
        self._create_variables()
        self._add_constraints()
        self._set_objective()
        # Ensure internal variables/constraints of the model are synchronized
        self.model.update()

    def _create_variables(self):
        """Create variables: x, delta, y, and w for linearization"""
        self.x = {}       # continuous expansion amount (slots) for each facility
        self.delta = {}   # binary: which segment chosen for each facility's expansion
        self.w = {}       # linearization: w[f,k] = x[f] * delta[f,k]
        
        # Create expansion decision variables for each facility
        for facility_id in self.facilities:
            # Get initial capacity of this facility
            facility_row = self.facility_data[self.facility_data["facility_id"] == facility_id]
            if facility_row.empty:
                continue
            initial_capacity = float(facility_row["initial_capacity_0_12"].values[0])
            
            # Only create expansion variables when initial capacity > 0
            if initial_capacity > 0:
                # Create continuous expansion variable
                self.x[facility_id] = self.model.addVar(vtype=GRB.INTEGER, lb=0.0, 
                                                      name=f"expand_{facility_id}")
                
                # Create delta binary variables for 3 segments
                for k in range(len(self.segment_bounds)):
                    self.delta[facility_id, k] = self.model.addVar(vtype=GRB.BINARY, 
                                                                 name=f"delta_{facility_id}_{k}")
                    # Create linearization variable w
                    self.w[facility_id, k] = self.model.addVar(vtype=GRB.CONTINUOUS, lb=0.0, 
                                                             name=f"w_{facility_id}_{k}")
            else:
                # Initial capacity is 0, use numeric 0 as placeholder
                self.x[facility_id] = 0.0
        
        # y variables for building new facilities (binary)
        self.y = {}
        for z in self.zips:
            if z in self.potential_locations and self.potential_locations[z]:
                for l in self.potential_locations[z]:
                    for t in self.facility_types:
                        self.y[z, l, t] = self.model.addVar(vtype=GRB.BINARY, name=f"build_{z}_{l}_{t}")

    def _add_constraints(self):
        """Add constraints: segment selection, linearization constraints, coverage constraints, distance constraints, etc."""
        # For each facility
        for facility_id in self.facilities:
            # Get initial capacity of this facility
            facility_row = self.facility_data[self.facility_data["facility_id"] == facility_id]
            if facility_row.empty:
                continue
            initial_capacity = float(facility_row["initial_capacity_0_12"].values[0])
            
            # Only add expansion constraints when initial capacity > 0
            if initial_capacity <= 0:
                continue
                
            # (A) Ensure one segment is selected
            self.model.addConstr(quicksum(self.delta[facility_id, k] for k in range(len(self.segment_bounds))) == 1,
                                 name=f"delta_sum_{facility_id}")
            
            # (B) Limit x to the selected segment range via big-M
            # And establish linearization constraints for w: w = x * delta
            M = initial_capacity * 0.2  # Big-M value is 20% of initial capacity
            # If M could be 0, set M to a small positive number
            if M <= 0:
                M = 1.0
                
            for k, (lower, upper) in enumerate(self.segment_bounds):
                # x >= lower * initial_capacity * delta
                self.model.addConstr(self.x[facility_id] >= lower * initial_capacity * self.delta[facility_id, k], 
                                     name=f"expand_lb_{facility_id}_{k}")
                # x <= upper * initial_capacity * delta + M * (1 - delta)
                self.model.addConstr(self.x[facility_id] <= upper * initial_capacity * self.delta[facility_id, k] + 
                                     M * (1 - self.delta[facility_id, k]),
                                     name=f"expand_ub_{facility_id}_{k}")
                # w linearization:
                # w <= x
                self.model.addConstr(self.w[facility_id, k] <= self.x[facility_id], 
                                     name=f"w_le_x_{facility_id}_{k}")
                # w <= M * delta
                self.model.addConstr(self.w[facility_id, k] <= M * self.delta[facility_id, k], 
                                     name=f"w_le_Md_{facility_id}_{k}")
                # w >= x - M*(1-delta)
                self.model.addConstr(self.w[facility_id, k] >= self.x[facility_id] - 
                                     M * (1 - self.delta[facility_id, k]), 
                                     name=f"w_ge_x_minus_M1d_{facility_id}_{k}")
                # w >= 0 (created via var lb)
        
        # Coverage constraints (by zip code)
        for z in self.zips:
            # Get data for this zip code
            zip_row = self.data[self.data["zip_code"] == z].iloc[0]
            current_012 = float(zip_row.get("existing_capacity_0_12", 0.0))
            req_012 = float(zip_row.get("min_required_0_12", 0.0))
            
            # Calculate total expansion from all facilities
            total_expansion = 0.0
            facilities_in_zip = self.facility_data[self.facility_data["zip_code"] == z]["facility_id"].tolist()
            for facility_id in facilities_in_zip:
                if facility_id in self.x:
                    # x[facility_id] could be a variable or a numeric value
                    if hasattr(self.x[facility_id], "X"):
                        total_expansion += self.x[facility_id]
                    else:
                        total_expansion += self.x[facility_id]
            
            # Capacity from new facilities
            new_capacity = 0
            if z in self.potential_locations and self.potential_locations[z]:
                new_capacity = quicksum(
                    self._get_capacity(t) * self.y[z, l, t]
                    for l in self.potential_locations[z]
                    for t in self.facility_types
                    if (z, l, t) in self.y
                )
            
            # 0-12 coverage constraint
            self.model.addConstr(current_012 + total_expansion + new_capacity >= req_012, 
                                 name=f"cov012_{z}")
            
            # 0-5 coverage constraint
            current_05 = float(zip_row.get("existing_capacity_0_5", 0.0))
            req_05 = float(zip_row.get("min_required_0_5", 0.0))
            new_05 = 0
            if z in self.potential_locations and self.potential_locations[z]:
                new_05 = quicksum(
                    self._get_05_capacity(t) * self.y[z, l, t]
                    for l in self.potential_locations[z]
                    for t in self.facility_types
                    if (z, l, t) in self.y
                )
            
            # Calculate total expansion for 0-5 (conservative approach)
            total_expansion_05 = 0.0
            for facility_id in facilities_in_zip:
                if facility_id in self.x:
                    # 0-5 expansion is same as total expansion (conservative assumption)
                    if hasattr(self.x[facility_id], "X"):
                        total_expansion_05 += self.x[facility_id]
                    else:
                        total_expansion_05 += self.x[facility_id]
            
            self.model.addConstr(current_05 + total_expansion_05 + new_05 >= req_05, 
                                 name=f"cov05_{z}")
        
        # Distance constraints: new-new and new-existing
        # new-new
        if self.distances is not None and not self.distances.empty:
            if "too_close" in self.distances.columns:
                close_pairs = self.distances[self.distances["too_close"] == True]
            else:
                close_pairs = self.distances
            for _, row in close_pairs.iterrows():
                z = int(row["zip_code"])
                loc1 = int(row["loc1_id"])
                loc2 = int(row["loc2_id"])
                if z not in self.potential_locations:
                    continue
                if loc1 not in self.potential_locations[z] or loc2 not in self.potential_locations[z]:
                    continue
                for t in self.facility_types:
                    if (z, loc1, t) in self.y and (z, loc2, t) in self.y:
                        self.model.addConstr(self.y[z, loc1, t] + self.y[z, loc2, t] <= 1,
                                             name=f"distance_new_{z}_{loc1}_{loc2}_{t}")
        
        # new-existing
        if self.too_close is not None and not self.too_close.empty:
            for _, row in self.too_close.iterrows():
                z = int(row["zip_code"])
                loc = int(row["location_id"])
                facility_id = int(row["facility_id"])
                if z not in self.potential_locations:
                    continue
                if loc not in self.potential_locations[z]:
                    continue
                for t in self.facility_types:
                    if (z, loc, t) in self.y:
                        self.model.addConstr(self.y[z, loc, t] == 0, 
                                            name=f"distance_exist_{z}_{loc}_{facility_id}_{t}")
                        
        # New constraint: at most one facility type can be built at each potential location (z, l)
        for z in self.zips:
            if z in self.potential_locations and self.potential_locations[z]:
                for l in self.potential_locations[z]:
                    self.model.addConstr(
                        quicksum(self.y[z, l, t] for t in self.facility_types if (z, l, t) in self.y) <= 1,
                        name=f"one_facility_per_location_{z}_{l}"
                    )

    def _set_objective(self):
        """Set the linearized objective function: build_cost + equip_cost + expand_cost (via w)"""
        # Build cost
        build_cost = quicksum(
            self._get_build_cost(t) * self.y[z, l, t]
            for z in self.zips
            if z in self.potential_locations and self.potential_locations[z]
            for l in self.potential_locations[z]
            for t in self.facility_types
            if (z, l, t) in self.y
        )
        
        # Equipment cost for 0-5 slots in new facilities: $100 per 0-5 slot
        equip_cost = quicksum(
            100.0 * self._get_05_capacity(t) * self.y[z, l, t]
            for z in self.zips
            if z in self.potential_locations and self.potential_locations[z]
            for l in self.potential_locations[z]
            for t in self.facility_types
            if (z, l, t) in self.y
        )
        
        # Expansion cost via w (linearized)
        expand_cost_terms = []
        for facility_id in self.facilities:
            # Get initial capacity of this facility
            facility_row = self.facility_data[self.facility_data["facility_id"] == facility_id]
            if facility_row.empty:
                continue
            initial_capacity = float(facility_row["initial_capacity_0_12"].values[0])
            
            # Only calculate expansion cost when initial capacity > 0
            if initial_capacity <= 0:
                continue
                
            for k in range(len(self.segment_bounds)):
                # cost factor per segment: (20,000 + factor * initial_capacity) * (x / initial_capacity)
                cost_factor = 20000.0 + self._get_cost_factor(k) * initial_capacity
                # term = cost_factor * w[facility_id,k] / initial_capacity
                expand_cost_terms.append((cost_factor / float(initial_capacity)) * self.w[facility_id, k])
        
        expand_cost = quicksum(expand_cost_terms) if expand_cost_terms else 0.0
        
        # Total objective
        self.model.setObjective(build_cost + equip_cost + expand_cost, GRB.MINIMIZE)

    def _get_cost_factor(self, segment_idx):
        """Cost factors for segments (corresponding to 200, 400, 1000 in PDF)"""
        cost_factors = [200, 400, 1000]
        return cost_factors[segment_idx]

    def _get_capacity(self, facility_type):
        """Total capacity (0-12)"""
        if facility_type == "small":
            return 100
        elif facility_type == "medium":
            return 200
        else:
            return 400

    def _get_05_capacity(self, facility_type):
        """Dedicated capacity for 0-5 age group"""
        if facility_type == "small":
            return 50
        elif facility_type == "medium":
            return 100
        else:
            return 200

    def _get_build_cost(self, facility_type):
        """Construction cost"""
        if facility_type == "small":
            return 65000.0
        elif facility_type == "medium":
            return 95000.0
        else:
            return 115000.0

    def solve(self, output_flag=1, mip_gap=0.05, time_limit=3600):
        """Solve the model and extract current solution even when non-optimal but feasible"""
        if self.model is None:
            self.build_model()
        self.model.setParam('OutputFlag', int(output_flag))
        self.model.setParam('MIPGap', float(mip_gap))
        self.model.setParam('TimeLimit', float(time_limit))
        self.model.optimize()
        status = self.model.status
        if status == GRB.OPTIMAL:
            return self._extract_results()
        elif status == GRB.TIME_LIMIT:
            # If time limit reached but there is a feasible solution, return that solution
            if self.model.SolCount > 0:
                print("Time limit reached — returning incumbent feasible solution.")
                return self._extract_results()
            else:
                print("Time limit reached — no feasible solution found.")
                return None
        elif status == GRB.INFEASIBLE:
            print("Model infeasible.")
            return None
        else:
            # Other statuses (feasible solution but not optimal)
            if self.model.SolCount > 0:
                print(f"Solver status {status} — returning incumbent solution.")
                return self._extract_results()
            print(f"Solver ended with status {status}. No solution returned.")
            return None

    def _extract_results(self):
        """Extract decision variable values from solver and construct DataFrame"""
        # First, get overall information for each zip code
        results = []
        for z in self.zips:
            row = self.data[self.data["zip_code"] == z].iloc[0]
            # Extract expansion information for all facilities in this zip code
            facilities_in_zip = self.facility_data[self.facility_data["zip_code"] == z]
            total_expand = 0.0
            expand_details = []
            
            for _, facility in facilities_in_zip.iterrows():
                facility_id = facility["facility_id"]
                # Extract x value
                x_val = 0.0
                if facility_id in self.x and hasattr(self.x[facility_id], "X"):
                    x_val = float(self.x[facility_id].X)
                elif facility_id in self.x:
                    x_val = float(self.x[facility_id])
                
                # Accumulate total expansion
                total_expand += x_val
                
                # Find selected segment
                seg = "N/A"
                for k in range(len(self.segment_bounds)):
                    if (facility_id, k) in self.delta and hasattr(self.delta[facility_id, k], "X") and self.delta[facility_id, k].X > 0.5:
                        seg = f"{int(self.segment_bounds[k][0]*100)}%-{int(self.segment_bounds[k][1]*100)}%"
                        break
                
                expand_details.append({
                    "facility_id": facility_id,
                    "expand": x_val,
                    "expand_segment": seg,
                    "initial_capacity": float(facility["initial_capacity_0_12"])
                })
            
            # Count of new facilities
            small = medium = large = 0
            if z in self.potential_locations and self.potential_locations[z]:
                for l in self.potential_locations[z]:
                    for t in self.facility_types:
                        if (z, l, t) in self.y and hasattr(self.y[z, l, t], "X") and self.y[z, l, t].X > 0.5:
                            if t == "small":
                                small += 1
                            elif t == "medium":
                                medium += 1
                            else:
                                large += 1
            
            total_new = small * self._get_capacity("small") + medium * self._get_capacity("medium") + large * self._get_capacity("large")
            results.append({
                "zip": z,
                "total_expand": total_expand,
                "expand_details": expand_details,
                "small": small,
                "medium": medium,
                "large": large,
                "total_new_capacity": total_new,
                "current_capacity_012": float(row.get("existing_capacity_0_12", 0.0)),
                "current_capacity_05": float(row.get("existing_capacity_0_5", 0.0)),
                "required_012": float(row.get("min_required_0_12", 0.0)),
                "required_05": float(row.get("min_required_0_5", 0.0))
            })
        
        return pd.DataFrame(results)

    def save_results(self, 
                     expansion_file="expansion_results.csv",
                     new_facility_file="new_facility_results.csv"):
        """Save results to two CSV files:
        - expansion_file: Details of expansion for existing facilities
        - new_facility_file: Locations and types of new facilities
        """
        results = self.solve()
        if results is None:
            print("No results to save.")
            return

        # ====== 1. Save expansion information ======
        expansion_data = []
        for _, row in results.iterrows():
            zip_code = row['zip']
            for facility in row['expand_details']:
                # Only save facilities with actual expansion (or decisions), even if expand=0 (as segment might be selected)
                expansion_data.append({
                    "zip_code": zip_code,
                    "facility_id": facility["facility_id"],
                    "initial_capacity_0_12": facility["initial_capacity"],
                    "expand_slots": facility["expand"],
                    "expand_segment": facility["expand_segment"]
                })

        if expansion_data:
            exp_df = pd.DataFrame(expansion_data)
            exp_df.to_csv(expansion_file, index=False)
            print(f"Expansion details for existing facilities saved to {expansion_file}")
        else:
            print("No expansion details to save.")

        # ====== 2. Save new facility information ======
        new_facility_data = []
        for z in self.zips:
            if z not in self.potential_locations or not self.potential_locations[z]:
                continue
            for l in self.potential_locations[z]:
                for t in self.facility_types:
                    if (z, l, t) in self.y:
                        var = self.y[z, l, t]
                        if hasattr(var, "X") and var.X > 0.5:
                            new_facility_data.append({
                                "zip_code": z,
                                "location_id": l,
                                "facility_type": t,
                                "capacity_0_12": self._get_capacity(t),
                                "capacity_0_5": self._get_05_capacity(t)
                            })

        if new_facility_data:
            new_df = pd.DataFrame(new_facility_data)
            new_df.to_csv(new_facility_file, index=False)
            print(f"New facility information saved to {new_facility_file}")
        else:
            print("No new facility information to save.")

        # ====== Print summary statistics ======
        total_expand = results['total_expand'].sum()
        total_new_facilities = results[['small', 'medium', 'large']].sum().sum()
        print("\nKey Statistics:")
        print(f"Total expanded capacity: {total_expand:,.0f} slots")
        print(f"Number of new facilities: {total_new_facilities:,.0f}")

        # Expansion segment distribution
        all_segments = []
        for _, row in results.iterrows():
            for facility in row['expand_details']:
                if facility['expand_segment'] != 'N/A':
                    all_segments.append(facility['expand_segment'])
        for seg in ["0%-10%", "10%-15%", "15%-20%"]:
            count = all_segments.count(seg)
            print(f"  {seg}: {count} facilities")

if __name__ == "__main__":
    print("\nStep 2: Solving optimization problem (per facility expansion)...")
    # Step 2: Solve optimization problem (per facility expansion)...
    planner = RealisticCapacityPlanner()
    # Optionally adjust solver output/parameters
    res = planner.solve(output_flag=1, mip_gap=0.05, time_limit=3600)
    if res is not None:
        print("\nOptimization results (top rows):")
        print(res.head())
        planner.save_results()
    print("\nKey Model Information:")
    print(f"Number of ZIP codes: {len(planner.zips)}")
    print(f"Number of facilities: {len(planner.facilities)}")